In [ ]:
# RAILPULSE | NB_03_ML_Delay_Forecast  (lakehouse attaché : LH_RailPulse_Silver)
# ---- Cellule 1 : jeu de caractéristiques commun (historique et horaire futur) ----
import mlflow, numpy as np, pandas as pd, datetime as dt
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from pyspark.sql import functions as F
fr = lambda n: f"{n:,}".replace(",", " ")   # 98 950 plutôt que 98,950

FEATURES = ["dep_hour", "day_of_week", "month", "is_peak", "distance_km", "cross_border",
            "weather_severity", "trainset_age", "load_factor"]
MODEL = "ML_RailPulse_Delay"
SILVER = "LH_RailPulse_Silver"   # chaque table est nommée avec son lakehouse : aucune dépendance au lakehouse par défaut

wx = spark.read.table(f"{SILVER}.silver_weather").select(F.to_date("weather_date").alias("service_date"), "station_code", "severity")
st = spark.read.table(f"{SILVER}.silver_stations").select("station_code", "country_code")
ts = spark.read.table(f"{SILVER}.silver_trainsets").select("trainset_id", "trainset_age")

def features(df):
    out = (df
        .join(wx.withColumnRenamed("station_code", "origin").withColumnRenamed("severity", "sev_o"), ["service_date", "origin"], "left")
        .join(wx.withColumnRenamed("station_code", "destination").withColumnRenamed("severity", "sev_d"), ["service_date", "destination"], "left")
        .join(st.withColumnRenamed("station_code", "origin").withColumnRenamed("country_code", "c_o"), "origin", "left")
        .join(st.withColumnRenamed("station_code", "destination").withColumnRenamed("country_code", "c_d"), "destination", "left")
        .join(ts, "trainset_id", "left")
        .withColumn("weather_severity", F.greatest(F.coalesce("sev_o", F.lit(0)), F.coalesce("sev_d", F.lit(0))))
        .withColumn("cross_border", (F.col("c_o") != F.col("c_d")).cast("int"))
        .withColumn("dep_hour", F.hour("scheduled_departure"))
        .withColumn("day_of_week", F.dayofweek("scheduled_departure"))
        .withColumn("month", F.month("scheduled_departure"))
        .withColumn("is_peak", F.col("dep_hour").isin(7, 8, 17, 18).cast("int")))
    for c in FEATURES:
        out = out.withColumn(c, F.col(c).cast("double"))
    return out

pax = spark.read.table(f"{SILVER}.silver_bookings").groupBy("trip_id").agg(F.sum("passengers").alias("pax"))
hist = features(spark.read.table(f"{SILVER}.silver_trips").filter("NOT is_cancelled").join(pax, "trip_id")
                .withColumn("load_factor", F.round(F.col("pax") / F.col("seats"), 3)))
pdf = hist.select("service_date", *FEATURES, F.col("arrival_delay_min").cast("double").alias("label")).toPandas()
cut = dt.date(2026, 8, 1)
train, test = pdf[pdf.service_date < cut], pdf[pdf.service_date >= cut]
print(f"Entraînement : {fr(len(train))} trains | test (août-septembre 2026) : {fr(len(test))} trains")

In [ ]:
# ---- Cellule 2 : entraînement suivi par MLflow et enregistrement du modèle ----
mlflow.set_experiment("EXP_RailPulse_Delay")
with mlflow.start_run(run_name="hgb_absolute_error") as run:
    params = dict(loss="absolute_error", max_iter=300, learning_rate=0.08, max_depth=6, random_state=42)
    model = HistGradientBoostingRegressor(**params).fit(train[FEATURES], train.label)
    pred = model.predict(test[FEATURES])
    mae = mean_absolute_error(test.label, pred)
    base = mean_absolute_error(test.label, np.full(len(test), train.label.median()))
    metrics = {"mae": mae, "r2": r2_score(test.label, pred), "mae_baseline": base, "gain_pct": 100 * (1 - mae / base)}
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    signature = infer_signature(train[FEATURES], model.predict(train[FEATURES]))
    mlflow.sklearn.log_model(model, "model", signature=signature, registered_model_name=MODEL)

version = max(int(v.version) for v in MlflowClient().search_model_versions(f"name='{MODEL}'"))
print(f"Modèle {MODEL} v{version} | MAE {mae:.2f} min contre {base:.2f} min sans modèle | gain {metrics['gain_pct']:.0f} %")

In [ ]:
# ---- Cellule 3 : PREDICT sur l'horaire des 14 prochains jours ----
from synapse.ml.predict import MLFlowTransformer

plan = features(spark.read.table(f"{SILVER}.silver_timetable_plan"))
scorer = MLFlowTransformer(inputCols=FEATURES, outputCol="predicted_delay_min", modelName=MODEL, modelVersion=version)
forecast = (scorer.transform(plan)
    .withColumn("predicted_delay_min", F.round("predicted_delay_min", 1))
    .withColumn("risk_level", F.when(F.col("predicted_delay_min") >= 6, "ÉLEVÉ")
                               .when(F.col("predicted_delay_min") >= 3, "MOYEN").otherwise("FAIBLE"))
    .withColumn("model_version", F.lit(version))
    .withColumn("_scored_at", F.current_timestamp())
    .select("trip_id", "service_date", "line_code", "origin", "destination", "scheduled_departure",
            "trainset_id", "weather_severity", "load_factor", "predicted_delay_min", "risk_level",
            "model_version", "_scored_at"))
forecast.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.ml_delay_forecast")

spark.createDataFrame([(run.info.run_id, version, float(mae), float(base), float(metrics["gain_pct"]))],
                      "run_id string, model_version int, mae double, mae_baseline double, gain_pct double") \
     .withColumn("_trained_at", F.current_timestamp()) \
     .write.mode("append").saveAsTable(f"{SILVER}.ml_model_metrics")
display(forecast.groupBy("risk_level").count())